# PodcastLens — Fine-Tuning on Custom Dataset

**Workflow:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run cells top-to-bottom (`Runtime → Run all`)
3. Upload your `dataset_clean.csv` when prompted
4. At the end, download `podcastlens_models.zip` and extract it into your project root

**dataset_clean.csv must have these columns:**
- `comment_text` — the comment
- `sentiment_label` — `positive` / `negative` / `neutral`
- `spam_label` — `spam` / `ham`

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU — go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn huggingface_hub

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # paste your HuggingFace token here (huggingface.co/settings/tokens)
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    login()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_BASE          = "/content/drive/MyDrive/podcastlens_models"
SENTIMENT_MODEL_DIR = f"{DRIVE_BASE}/sentiment-finetuned"
SPAM_MODEL_DIR      = f"{DRIVE_BASE}/spam-finetuned"

for d in (SENTIMENT_MODEL_DIR, SPAM_MODEL_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

print("Drive mounted. Model dirs ready:")

---
## Section 1 — Load & Inspect Dataset

In [ ]:
from google.colab import files
import pandas as pd
import io

print("Upload your dataset_clean.csv file:")
uploaded = files.upload()
filename = next(iter(uploaded))
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]), encoding="utf-8-sig")

print(f"Loaded: {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
print("Columns:", list(df_raw.columns))
df_raw.head(5)

In [ ]:
required = {"comment_text", "sentiment_label", "spam_label"}
missing_cols = required - set(df_raw.columns)
if missing_cols:
    raise ValueError(f"CSV is missing required columns: {missing_cols}")

print("=== Dataset Info ===")
print(f"Total rows         : {len(df_raw):,}")
print(f"Missing comment_text: {df_raw['comment_text'].isna().sum()}")
print(f"Missing sentiment  : {df_raw['sentiment_label'].isna().sum()}")
print(f"Missing spam       : {df_raw['spam_label'].isna().sum()}")
print()
print("=== Raw sentiment values ===")
print(df_raw["sentiment_label"].value_counts(dropna=False))
print()
print("=== Raw spam values ===")
print(df_raw["spam_label"].value_counts(dropna=False))

---
## Section 2 — Preprocessing

In [ ]:
df = df_raw.copy()

# Normalize text labels
df["comment_text"]   = df["comment_text"].astype(str).str.strip()
df["sentiment_label"] = df["sentiment_label"].astype(str).str.strip().str.lower()
df["spam_label"]      = df["spam_label"].astype(str).str.strip().str.lower()

# Map to standard values
SENTIMENT_VALID = {"positive", "negative", "neutral"}
SPAM_VALID      = {"spam", "ham"}

df["sentiment_label"] = df["sentiment_label"].where(df["sentiment_label"].isin(SENTIMENT_VALID))
df["spam_label"]      = df["spam_label"].where(df["spam_label"].isin(SPAM_VALID))

# Drop rows with empty comment text
df = df[df["comment_text"].str.len() > 0]

print(f"Rows after text cleanup: {len(df):,}")
print()
print("=== Cleaned sentiment distribution ===")
print(df["sentiment_label"].value_counts(dropna=False))
print()
print("=== Cleaned spam distribution ===")
print(df["spam_label"].value_counts(dropna=False))

In [ ]:
# Numeric label maps (required by transformers Trainer)
SENTIMENT_MAP = {"negative": 0, "neutral": 1, "positive": 2}
SPAM_MAP      = {"ham": 0, "spam": 1}

df["sentiment_int"] = df["sentiment_label"].map(SENTIMENT_MAP)
df["spam_int"]      = df["spam_label"].map(SPAM_MAP)

# Separate dataframes — only keep rows that have each label
df_sentiment = df.dropna(subset=["sentiment_int"]).reset_index(drop=True)
df_spam      = df.dropna(subset=["spam_int"]).reset_index(drop=True)

df_sentiment["sentiment_int"] = df_sentiment["sentiment_int"].astype(int)
df_spam["spam_int"]           = df_spam["spam_int"].astype(int)

print(f"Sentiment dataset : {len(df_sentiment):,} labeled rows")
print(f"Spam dataset      : {len(df_spam):,} labeled rows")

---
## Section 3 — Shuffle & Split (80 / 10 / 10)

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

def make_splits(df_in, label_col):
    df_in = df_in.sample(frac=1, random_state=SEED).reset_index(drop=True)
    train, temp = train_test_split(df_in, test_size=0.20, random_state=SEED,
                                   stratify=df_in[label_col])
    val, test   = train_test_split(temp,  test_size=0.50, random_state=SEED,
                                   stratify=temp[label_col])
    return train, val, test

sent_train, sent_val, sent_test = make_splits(df_sentiment, "sentiment_int")
spam_train, spam_val, spam_test = make_splits(df_spam,      "spam_int")

print("=== Sentiment splits ===")
for name, s in [("train", sent_train), ("val", sent_val), ("test", sent_test)]:
    dist = s["sentiment_label"].value_counts().to_dict()
    print(f"  {name:5}: {len(s):>5} rows  {dist}")

print()
print("=== Spam splits ===")
for name, s in [("train", spam_train), ("val", spam_val), ("test", spam_test)]:
    dist = s["spam_label"].value_counts().to_dict()
    print(f"  {name:5}: {len(s):>5} rows  {dist}")

---
## Section 4 — Sentiment Fine-tuning
Model: `cardiffnlp/twitter-roberta-base-sentiment-latest`
Labels: `0=negative  1=neutral  2=positive`

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import classification_report

SENT_BASE   = "cardiffnlp/twitter-roberta-base-sentiment-latest"
SENT_LABELS = ["negative", "neutral", "positive"]

sent_tokenizer = AutoTokenizer.from_pretrained(SENT_BASE)
sent_model     = AutoModelForSequenceClassification.from_pretrained(
    SENT_BASE, num_labels=3, ignore_mismatched_sizes=True
)

def tok_sent(batch):
    return sent_tokenizer(batch["comment_text"], truncation=True, max_length=128)

def df_to_ds(df_in, label_col):
    ds = Dataset.from_dict({"comment_text": df_in["comment_text"].tolist(),
                             "labels":      df_in[label_col].tolist()})
    return ds.map(tok_sent, batched=True)

sent_tr = df_to_ds(sent_train, "sentiment_int")
sent_vl = df_to_ds(sent_val,   "sentiment_int")
sent_te = df_to_ds(sent_test,  "sentiment_int")

for ds in (sent_tr, sent_vl, sent_te):
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

sent_args = TrainingArguments(
    output_dir          = SENTIMENT_MODEL_DIR,
    num_train_epochs    = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 2e-5,
    weight_decay        = 0.01,
    warmup_ratio        = 0.1,
    eval_strategy       = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    logging_steps       = 20,
    report_to           = "none",
    fp16                = torch.cuda.is_available(),
)

sent_trainer = Trainer(
    model         = sent_model,
    args          = sent_args,
    train_dataset = sent_tr,
    eval_dataset  = sent_vl,
    tokenizer     = sent_tokenizer,
    data_collator = DataCollatorWithPadding(sent_tokenizer),
)

print("Training sentiment model...")
sent_trainer.train()
sent_trainer.save_model(SENTIMENT_MODEL_DIR)
sent_tokenizer.save_pretrained(SENTIMENT_MODEL_DIR)
print(f"Saved to {SENTIMENT_MODEL_DIR}")

In [ ]:
print("=== Sentiment — Test Set Evaluation ===")
preds_out = sent_trainer.predict(sent_te)
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids
print(classification_report(labels, preds, target_names=SENT_LABELS))

---
## Section 5 — Spam Fine-tuning
Model: `distilbert-base-uncased`
Labels: `0=ham  1=spam`

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

SPAM_BASE = "distilbert-base-uncased"

spam_tokenizer = AutoTokenizer.from_pretrained(SPAM_BASE)
spam_model     = AutoModelForSequenceClassification.from_pretrained(SPAM_BASE, num_labels=2)

def tok_spam(batch):
    return spam_tokenizer(batch["comment_text"], truncation=True, max_length=128)

def df_to_spam_ds(df_in, label_col):
    ds = Dataset.from_dict({"comment_text": df_in["comment_text"].tolist(),
                             "labels":      df_in[label_col].tolist()})
    return ds.map(tok_spam, batched=True)

spam_tr = df_to_spam_ds(spam_train, "spam_int")
spam_vl = df_to_spam_ds(spam_val,   "spam_int")
spam_te = df_to_spam_ds(spam_test,  "spam_int")

for ds in (spam_tr, spam_vl, spam_te):
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

spam_args = TrainingArguments(
    output_dir          = SPAM_MODEL_DIR,
    num_train_epochs    = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 2e-5,
    weight_decay        = 0.01,
    warmup_ratio        = 0.1,
    eval_strategy       = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    logging_steps       = 20,
    report_to           = "none",
    fp16                = torch.cuda.is_available(),
)

spam_trainer = Trainer(
    model         = spam_model,
    args          = spam_args,
    train_dataset = spam_tr,
    eval_dataset  = spam_vl,
    tokenizer     = spam_tokenizer,
    data_collator = DataCollatorWithPadding(spam_tokenizer),
)

print("Training spam classifier...")
spam_trainer.train()
spam_trainer.save_model(SPAM_MODEL_DIR)
spam_tokenizer.save_pretrained(SPAM_MODEL_DIR)
print(f"Saved to {SPAM_MODEL_DIR}")

In [ ]:
print("=== Spam — Test Set Evaluation ===")
preds_out = spam_trainer.predict(spam_te)
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids
print("Confusion matrix:")
print(confusion_matrix(labels, preds))
print(f"F1 macro: {f1_score(labels, preds, average='macro'):.4f}")
print(classification_report(labels, preds, target_names=["ham", "spam"]))

---
## Section 6 — Package & Download

This cell:
1. Creates `models_config.json` with **local relative paths** so the app works after extraction
2. Zips `models/sentiment-finetuned/`, `models/spam-finetuned/`, and `models_config.json`
3. Downloads `podcastlens_models.zip`

**After downloading:** extract the zip into your project root, then restart `uvicorn main:app`.

In [ ]:
import json, shutil
from datetime import datetime
from google.colab import files

EXPORT_DIR = "/content/podcastlens_export"

# Build export folder with the exact structure the local app expects:
#   models/sentiment-finetuned/
#   models/spam-finetuned/
#   models_config.json
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(f"{EXPORT_DIR}/models", exist_ok=True)

sentiment_ok = Path(SENTIMENT_MODEL_DIR).is_dir() and any(Path(SENTIMENT_MODEL_DIR).iterdir())
spam_ok      = Path(SPAM_MODEL_DIR).is_dir()      and any(Path(SPAM_MODEL_DIR).iterdir())

if sentiment_ok:
    shutil.copytree(SENTIMENT_MODEL_DIR, f"{EXPORT_DIR}/models/sentiment-finetuned")
    print("Copied sentiment model.")

if spam_ok:
    shutil.copytree(SPAM_MODEL_DIR, f"{EXPORT_DIR}/models/spam-finetuned")
    print("Copied spam model.")

if not sentiment_ok and not spam_ok:
    print("ERROR: No models were saved. Check errors in training cells.")
else:
    # Write config with LOCAL relative paths — works after zip extraction
    config = {
        "sentiment_model": "./models/sentiment-finetuned",
        "spam_model":      "./models/spam-finetuned",
        "fine_tuned":      True,
        "fine_tuned_at":   datetime.utcnow().isoformat(),
    }
    with open(f"{EXPORT_DIR}/models_config.json", "w") as f:
        json.dump(config, f, indent=2)
    print("Written models_config.json with local paths.")

    zip_path = "/content/podcastlens_models"
    shutil.make_archive(zip_path, "zip", EXPORT_DIR)
    print(f"Archive size: {Path(zip_path+'.zip').stat().st_size / 1e6:.1f} MB")
    print("Downloading...")
    files.download(zip_path + ".zip")